# Test a portable chat client

You are building a help-desk assistant that talks to any OpenAI-compatible `/v1/chat/completions` server: a hosted API, Ollama, `llama-server` or vLLM. Every network response here is a recorded fixture, so failures are repeatable and no model download or API key is needed.

The lesson exercises gave you single-step pieces. Each task below extends them: multi-turn payloads and reply parsing, a full retry schedule, a byte-level stream parser, a failure-aware runtime comparison and finally the whole adapter.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.

## Given: lesson solutions and fixture helpers

Run this cell first. It holds the solutions to the lesson exercises (`retry_delay`, `collect_sse`, `summarize_runs`) for you to build on, plus two helpers that make recorded streams:

- `sse_stream(*pieces, finish="stop", done=True)` returns the raw bytes a server would send for those text pieces.
- `chop(data, size)` cuts bytes into network reads of `size` bytes, so lines and even characters get split across reads.

In [ ]:
import json
from math import ceil

def retry_delay(status, attempt, max_attempts, base, remaining, retry_after=0):
    transient = status in {408, 429, 500, 502, 503, 504}
    if not transient or attempt >= max_attempts:
        return None
    delay = max(base * 2 ** (attempt - 1), retry_after)
    return delay if delay < remaining else None

def collect_sse(lines):
    parts, finish = [], None
    for line in lines:
        if not line.startswith("data:"):
            continue
        data = line[5:].strip()
        if data == "[DONE]":
            return "".join(parts), "truncated" if finish == "length" else "complete"
        choice = json.loads(data)["choices"][0]
        parts.append(choice["delta"].get("content") or "")
        finish = choice.get("finish_reason") or finish
    return "".join(parts), "interrupted"

def summarize_runs(seconds):
    successful = sorted(value for value in seconds if value is not None)
    p90 = successful[ceil(0.9 * len(successful)) - 1] if successful else None
    return p90, len(seconds) - len(successful)

def sse_stream(*pieces, finish="stop", done=True):
    def event(delta, reason=None):
        return "data: " + json.dumps({"choices": [{"index": 0, "delta": delta, "finish_reason": reason}]}, ensure_ascii=False) + "\n\n"
    text = event({"role": "assistant", "content": ""}) + ": keep-alive\n\n"
    text += "".join(event({"content": piece}) for piece in pieces)
    text += event({}, finish)
    if done:
        text += "data: [DONE]\n\n"
    return text.encode("utf-8")

def chop(data, size):
    return [data[i:i + size] for i in range(0, len(data), size)]

print(chop(sse_stream("Hi"), 40)[:3])

## 1. Multi-turn payload and reply reader

The lesson payload had one system and one user message. A help desk keeps a conversation going.

Write `chat_payload(model, system, history, user, max_tokens, temperature=0.2)`. `history` is a list of earlier `(user_text, assistant_text)` pairs, oldest first. Return a chat-completions body with `model`, `messages` (system first, then each earlier pair as a user and an assistant message, then the new user message), `max_tokens`, `temperature` and `"stream": False`. Keep all text verbatim.

Write `read_reply(response)` for a non-streaming response. Return `(text, finish_reason, total_tokens)` from `choices[0]` and `usage`. `usage` is optional: return `None` for `total_tokens` when it is missing.

In [ ]:
def chat_payload(model, system, history, user, max_tokens, temperature=0.2):
    raise NotImplementedError

def read_reply(response):
    raise NotImplementedError

In [ ]:
p = chat_payload("m", "Be brief.", [("Hi", "Hello!"), ("Order 12?", "Shipped.")], "When?", 64)
assert [m["role"] for m in p["messages"]] == ["system", "user", "assistant", "user", "assistant", "user"], "System first, then each (user, assistant) pair in order, then the new user message"
assert p["messages"][3] == {"role": "user", "content": "Order 12?"}, "Earlier turns keep their text and order"
assert p["messages"][-1] == {"role": "user", "content": "When?"}, "The new user message goes last"
assert (p["model"], p["max_tokens"], p["temperature"], p["stream"]) == ("m", 64, 0.2, False), "Use the chat-completions fields model, max_tokens, temperature and stream"
assert chat_payload("m", "s", [], "  Québec  ", 8, 0)["messages"] == [{"role": "system", "content": "s"}, {"role": "user", "content": "  Québec  "}], "An empty history gives just system and user, with text unchanged"
r = {"choices": [{"index": 0, "message": {"role": "assistant", "content": "Shipped Monday."}, "finish_reason": "stop"}],
     "usage": {"prompt_tokens": 30, "completion_tokens": 4, "total_tokens": 34}}
assert read_reply(r) == ("Shipped Monday.", "stop", 34), "Read choices[0].message.content, choices[0].finish_reason and usage.total_tokens"
cut = {"choices": [{"index": 0, "message": {"role": "assistant", "content": "Ship"}, "finish_reason": "length"}]}
assert read_reply(cut) == ("Ship", "length", None), "usage is optional, and a length finish must be passed through so callers can see the cut"
print("OK")

## 2. A whole retry schedule

The lesson function decides one retry. Now replay a run of failures against a deadline.

Write `retry_schedule(failures, max_attempts, base, deadline, attempt_seconds)`. `failures` lists the `(status, retry_after)` of consecutive failed attempts, first attempt first. Each attempt takes `attempt_seconds`. Start the clock at zero. After each failed attempt, add its duration, then call the given `retry_delay` with the attempt number and the time left before `deadline`. If it returns `None`, stop. Otherwise record the delay and add it to the clock. Return the list of delays actually slept.

In [ ]:
def retry_schedule(failures, max_attempts, base, deadline, attempt_seconds):
    raise NotImplementedError

In [ ]:
assert retry_schedule([(503, 0)] * 3, 4, 0.5, 10, 1) == [0.5, 1, 2], "Backoff doubles: 0.5, 1, 2 seconds"
assert retry_schedule([(503, 0)] * 3, 3, 0.5, 10, 1) == [0.5, 1], "The third attempt is the last of three, so it gets no retry"
assert retry_schedule([(429, 3), (503, 0)], 5, 0.5, 5, 1) == [3], "Honour the 3 s hint, then the clock reaches the 5 s deadline"
assert retry_schedule([(401, 0), (503, 0)], 5, 0.5, 30, 1) == [], "A credential error stops the schedule at once"
assert retry_schedule([], 3, 0.5, 10, 1) == [], "No failures means no sleeps"
print("OK")

## 3. Parse a stream from raw bytes

The lesson parser took tidy lines. A real client gets network reads of bytes, and a read can end anywhere: in the middle of `data: {...}`, between `\r` and `\n`, or inside a multi-byte character such as `é`.

Write `parse_sse_bytes(chunks)` for a list of `bytes` reads. Buffer bytes until you have a whole line (ending in `\n`), decode each complete line as UTF-8, drop a trailing `\r`, and pass the lines to the given `collect_sse`. Return its `(text, status)`. Bytes left without a final newline when the reads run out are an unfinished line: drop them.

In [ ]:
def parse_sse_bytes(chunks):
    raise NotImplementedError

In [ ]:
stream = sse_stream("Bonjour ", "Québec", "!")
assert parse_sse_bytes([stream]) == ("Bonjour Québec!", "complete"), "One read holding the whole stream"
for size in (1, 3, 7, 50):
    assert parse_sse_bytes(chop(stream, size)) == ("Bonjour Québec!", "complete"), f"Reads of {size} bytes split lines and the two bytes of é: buffer bytes, decode whole lines"
crlf = stream.replace(b"\n", b"\r\n")
assert parse_sse_bytes(chop(crlf, 5)) == ("Bonjour Québec!", "complete"), "Lines may end in CRLF: strip the trailing carriage return"
assert parse_sse_bytes(chop(sse_stream("Step 1", ", step", finish="length"), 9)) == ("Step 1, step", "truncated"), "finish_reason length then [DONE] is a truncated answer"
assert parse_sse_bytes(chop(sse_stream("Hel", "lo", done=False), 6)) == ("Hello", "interrupted"), "No [DONE] means interrupted"
cut = stream[:stream.index(b"Qu") + 4]
assert parse_sse_bytes(chop(cut, 4)) == ("Bonjour ", "interrupted"), "A connection lost mid-line leaves an unfinished line: drop it instead of parsing half a JSON object"
assert parse_sse_bytes([]) == ("", "interrupted"), "No bytes at all is an interrupted stream"
print("OK")

## 4. Compare hosted and local runs

You ran the same prompts on a hosted API and on a local runtime. For each label you have a list of request times in seconds (`None` for a failed request) and a list of quality scores between 0 and 1, one per request in the same order. A failed request's score is meaningless, so it counts as 0.

Write `compare_runtimes(hosted, local, hosted_quality, local_quality)`. Return a dictionary with keys `"hosted"` and `"local"`, each holding `{"p90": ..., "failure_rate": ..., "quality": ...}`, plus `"preferred"`:

- `p90` is the nearest-rank p90 of successful times (use the given `summarize_runs`).
- `failure_rate` is failures divided by all requests.
- `quality` is the sum of scores of successful requests divided by **all** requests, so failures pull it down.
- `"preferred"` is the label with higher `quality`, or `"tie"` if they are equal.

In [ ]:
def compare_runtimes(hosted, local, hosted_quality, local_quality):
    raise NotImplementedError

In [ ]:
from math import isclose
r = compare_runtimes([0.8, 1.2, None, 0.9], [2.0, 2.5, 3.0, 2.2],
                     [0.9, 0.8, 1.0, 1.0], [0.7, 0.8, 0.6, 0.7])
assert r["hosted"]["p90"] == 1.2 and r["local"]["p90"] == 3.0, "p90 uses successful times only"
assert isclose(r["hosted"]["failure_rate"], 0.25) and r["local"]["failure_rate"] == 0, "failure_rate is failures over all requests"
assert isclose(r["hosted"]["quality"], 0.675), "Hosted quality is (0.9 + 0.8 + 1.0) / 4: the failed request earns 0 even though its score was 1.0"
assert isclose(r["local"]["quality"], 0.7), "Local quality is 2.8 / 4"
assert r["preferred"] == "local", "Hosted scores 0.9 on the requests that succeed, but one failure in four drags it below local"
r2 = compare_runtimes([1.0, 1.0], [None, None], [0.8, 0.6], [1.0, 1.0])
assert r2["local"] == {"p90": None, "failure_rate": 1.0, "quality": 0.0}, "A runtime that always fails has no p90 and earns no quality"
assert r2["preferred"] == "hosted", "Any working runtime beats one that always fails"
assert compare_runtimes([1.0], [2.0], [0.5], [0.5])["preferred"] == "tie", "Equal quality is a tie"
print("OK")

## 5. Connect the adapter steps on a replayable request

Implement `replay_request(user, attempts)`. Build the payload with your `chat_payload` for model `"fixture"`, system text `"Be brief."`, no history and `max_tokens` 32. Then work through `attempts`, a list of `(status, chunks)` pairs where `chunks` is a list of byte reads, using **at most two attempts** in total:

- On status 200, parse the chunks with `parse_sse_bytes` and return `(payload, text, complete)` straight away, where `complete` is `True` only for the status `"complete"`. A truncated or interrupted 200 is returned as it is, not retried.
- On any other status, ask the given `retry_delay` with `max_attempts=2`, `base=0.25` and `remaining=2`. If it returns `None`, return `(payload, "", False)`. Otherwise go on to the next attempt.
- If the list runs out, return `(payload, "", False)`.

This is a fixture replay, so do not sleep or call any endpoint.

In [ ]:
def replay_request(user, attempts):
    raise NotImplementedError

In [ ]:
payload, text, complete = replay_request("Hi", [(503, []), (200, chop(sse_stream("Hello"), 16))])
assert payload["messages"][-1] == {"role": "user", "content": "Hi"}, "The payload ends with the user message"
assert payload["max_tokens"] == 32 and payload["model"] == "fixture", "Use model fixture and max_tokens 32"
assert (text, complete) == ("Hello", True), "A 503 then a clean stream should succeed on the second attempt"
assert replay_request("Hi", [(503, []), (503, []), (200, chop(sse_stream("x"), 16))])[1:] == ("", False), "At most two attempts: the third (successful) attempt must never be used"
assert replay_request("Hi", [(429, []), (200, chop(sse_stream("ok"), 16))])[1:] == ("ok", True), "A rate limit is retried once"
assert replay_request("Hi", [(200, chop(sse_stream("partial", done=False), 16)), (200, chop(sse_stream("again"), 16))])[1:] == ("partial", False), "An interrupted 200 is returned as incomplete, not silently retried"
assert replay_request("Hi", [(200, chop(sse_stream("cut", finish="length"), 16))])[1:] == ("cut", False), "A truncated answer is not complete"
assert replay_request("Hi", [(401, []), (200, chop(sse_stream("unexpected"), 16))])[1:] == ("", False), "A credential error is not retried"
assert replay_request("Hi", [])[1:] == ("", False), "No attempts means no answer"
print("OK")

## Interpretation

Draw the interface between the caller and the transport: which function owns the payload, the retry decision, the byte buffering and the final status. Explain what your client reports if a provider disconnects after sending two deltas, and how that differs from a `finish_reason` of `length`. Using task 4, say which numbers (latency percentile, failure rate, failure-adjusted quality, cost per 1,000 requests) would justify switching from hosted to local, and what extra quality checks you would want first.

**Your answer**